# U10-02 · Gráficos interactivos con Plotly y muchos datos

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 10, apartados 10.6 y 10.7*

Ejecuta cada celda con **Mayús + Intro**. Para que funcione en el navegador, la flota simulada tiene **50 máquinas** en lugar de 200: las máquinas 0-24 son las del sensor descalibrado (en el libro, 0-99). Las figuras se parecerán a las del libro, pero no serán idénticas.

In [ ]:
import os, sys
if sys.platform == "emscripten":
    # En el navegador, DuckDB no lee la carpeta de JupyterLite:
    # se trabaja en una carpeta temporal en memoria.
    os.chdir("/tmp")
%pip install -q duckdb pyarrow seaborn plotly
import duckdb, seaborn, plotly
print("DuckDB", duckdb.__version__, "· Seaborn", seaborn.__version__,
      "· Plotly", plotly.__version__)

## 0. Datos y plantilla

El generador de la unidad 9 (50 máquinas) y la plantilla `estilo_empresa` del apartado 10.3.

In [ ]:
import numpy as np
import pandas as pd

def generar_lecturas(maquinas=50, horas=24 * 365, semilla=0):
    """Lecturas horarias simuladas de una flota durante un año."""
    rng = np.random.default_rng(semilla)
    ritmo = rng.uniform(0.5, 1.5, maquinas)      # desgaste por hora
    desgaste = rng.uniform(0, 300, maquinas)     # desde la revisión
    filas = []
    for h in range(horas):
        desgaste += ritmo
        temp = 60 + 0.01 * desgaste + rng.normal(0, 2, maquinas)
        if h >= 24 * 273:              # desde octubre, media flota
            temp[: maquinas // 2] += 3  # con sensor descalibrado
        vib = 2 + 0.004 * desgaste + rng.normal(0, 0.5, maquinas)
        kw = 40 + 0.5 * (temp - 60) + rng.normal(0, 3, maquinas)
        z = -9 + 0.006 * desgaste + 0.8 * (vib - 2)
        averia = rng.random(maquinas) < 1 / (1 + np.exp(-z))
        filas.append((np.full(maquinas, h), np.arange(maquinas),
                      temp, vib, kw, desgaste.copy(), averia))
        desgaste[averia] = 0                      # se repara
    h, m, t, v, k, d, a = map(np.concatenate, zip(*filas))
    df = pd.DataFrame({
        "fecha": pd.Timestamp("2025-01-01") + pd.to_timedelta(h, "h"),
        "maquina": m, "temperatura": t, "vibracion": v,
        "consumo_kw": k, "horas_desde_revision": d, "averia": a})
    # objetivo: ¿se averiará la máquina en las próximas 24 horas?
    df = df.sort_values(["maquina", "fecha"], ignore_index=True)
    def en_24h(s):                               # ¿avería en las 24 h
        futuro = s[::-1].rolling(24, 1).max()[::-1]   # siguientes?
        return futuro.shift(-1, fill_value=0)
    proxima = df.groupby("maquina")["averia"].transform(en_24h)
    df["averia_24h"] = proxima.astype(bool)
    return df

In [ ]:
lecturas = generar_lecturas()
lecturas = lecturas.astype({"maquina": "int16", "temperatura": "float32",
    "vibracion": "float32", "consumo_kw": "float32",
    "horas_desde_revision": "float32"})
lecturas.to_parquet("lecturas.parquet", index=False)
print(len(lecturas), "lecturas guardadas en lecturas.parquet")

In [ ]:
import matplotlib as mpl
from cycler import cycler

def estilo_empresa():
    """Aspecto común para todos los gráficos de la empresa."""
    mpl.rcParams.update({
        "font.family": "sans-serif", "font.size": 7,
        "figure.figsize": (4.4, 2.6),          # 11 cm de ancho
        "axes.prop_cycle": cycler(color=[
            "#1F5FA8", "#E08A1E", "#7FA7D6", "#14304F", "#9AA5B1"]),
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": "#E3E8EE",
        "lines.linewidth": 1.2, "lines.markersize": 3.5,
        "grid.linewidth": 0.6, "axes.axisbelow": True,
        "axes.titlesize": 7.5, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "legend.frameon": False,
        "savefig.bbox": "tight", "pdf.fonttype": 42})

## 1. Un gráfico interactivo

In [ ]:
import os
import duckdb
import plotly.express as px

resumen = duckdb.sql("""
    SELECT maquina,
        count_if(averia)::INTEGER AS averias,
        round(avg(vibracion), 2) AS vibracion_media,
        round(avg(temperatura) FILTER (month(fecha) >= 10)
              - avg(temperatura) FILTER (month(fecha) <= 9), 1)
              AS salto_temp,
        CASE WHEN maquina < 25 THEN 'afectada'
             ELSE 'sin cambio' END AS grupo
    FROM 'lecturas.parquet' GROUP BY maquina
""").df()

fig = px.scatter(resumen, x="vibracion_media", y="averias",
                 color="grupo", hover_name="maquina",
                 hover_data=["salto_temp"],
                 labels={"vibracion_media": "Vibración media",
                         "averias": "Averías en el año",
                         "salto_temp": "Salto de temperatura"},
                 title="Averías y vibración de cada máquina")
for modo in [True, "cdn"]:        # plotly.js dentro o enlazado
    ruta = f"flota_{'completo' if modo is True else 'cdn'}.html"
    fig.write_html(ruta, include_plotlyjs=modo)
    print(ruta, f"{os.path.getsize(ruta) / 1024:.0f} KB")
print(resumen[["averias", "vibracion_media"]].corr().iloc[0, 1]
      .round(2))

Para verlo aquí mismo (pasa el ratón por los puntos y pulsa en la leyenda):

In [ ]:
import html
from IPython.display import HTML

def ver(figura, alto=480):
    """Muestra un gráfico de Plotly dentro del cuaderno."""
    pagina = html.escape(figura.to_html(include_plotlyjs="cdn"))
    return HTML(f'<iframe srcdoc="{pagina}" style="width:100%;'
                f'height:{alto}px;border:0"></iframe>')

ver(fig)

**Prueba.** Añade `size="salto_temp"` a `px.scatter`.

## 2. Muchos puntos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

estilo_empresa()
d = pd.read_parquet("lecturas.parquet",
                    columns=["horas_desde_revision", "vibracion"])
x, y = d["horas_desde_revision"], d["vibracion"]

fig, ejes = plt.subplots(1, 3, figsize=(4.4, 1.8), sharey=True)
ejes[0].scatter(x, y, s=0.3)
ejes[1].scatter(x, y, s=0.1, alpha=0.01, linewidths=0)
hb = ejes[2].hexbin(x, y, gridsize=40, bins="log", cmap="viridis")
titulos = ["Todos", "Transparencia", "Hexágonos (log)"]
for eje, titulo in zip(ejes, titulos):
    eje.set(title=titulo, xlabel="Horas")
    for capa in eje.collections:
        capa.set_rasterized(True)      # los puntos, como imagen
ejes[0].set_ylabel("Vibración")
fig.savefig("muchos.pdf", dpi=200)
print(len(d), "puntos; mayor celda:", int(hb.get_array().max()),
      "lecturas")

**Ejercicio.** Prueba `gridsize=20` y `gridsize=80` en los hexágonos, y quita `bins="log"`. ¿Qué se pierde?

> Shneiderman, B. (1996). The eyes have it. *IEEE Symposium on Visual Languages*, 336–343. https://doi.org/10.1109/VL.1996.545307